# Airline Delay Dataset Cleaning — Raw to Clean Wide CSV

Notebook này dùng để chạy trên **Google Colab**. Mục tiêu: đọc file dữ liệu gốc `Airline_Delay_Cause.csv`, kiểm tra chất lượng dữ liệu và xuất ra **file clean dạng wide**.

Output chính:
- `Airline_Delay_Cause_clean_wide.csv`: dataset đã làm sạch, vẫn giữ cấu trúc wide.
- `cleaning_summary.json`: log tóm tắt quá trình kiểm tra và làm sạch.

Lưu ý: notebook này **không tạo long format**, vì mục tiêu chỉ là làm sạch từ raw sang clean wide.


In [ ]:
# 0. Import thư viện
import os
import re
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)


## 1. Upload và đọc file dữ liệu gốc

Khi chạy trên Colab, chạy cell dưới đây rồi upload file CSV gốc. Nếu file đã có sẵn trong thư mục làm việc, notebook sẽ tự tìm file `.csv` đầu tiên.


In [ ]:
# 1. Upload file CSV gốc trên Google Colab
# Nếu chạy local/Jupyter thường, có thể bỏ qua phần upload và đặt RAW_FILE = 'ten_file.csv'

RAW_FILE = None  # ví dụ: RAW_FILE = 'Airline_Delay_Cause.csv'

try:
    from google.colab import files
    if RAW_FILE is None:
        print('Hãy upload file CSV gốc...')
        uploaded = files.upload()
        if uploaded:
            RAW_FILE = list(uploaded.keys())[0]
except Exception:
    pass

if RAW_FILE is None:
    csv_files = [f for f in os.listdir('.') if f.lower().endswith('.csv')]
    if len(csv_files) == 0:
        raise FileNotFoundError('Không tìm thấy file CSV. Hãy upload file hoặc đặt RAW_FILE = tên file.')
    RAW_FILE = csv_files[0]

print('Đang đọc file:', RAW_FILE)

# Đọc CSV linh hoạt hơn: thử utf-8 trước, nếu lỗi thì thử latin1
try:
    raw_df = pd.read_csv(RAW_FILE, encoding='utf-8')
except UnicodeDecodeError:
    raw_df = pd.read_csv(RAW_FILE, encoding='latin1')

print('Kích thước dữ liệu gốc:', raw_df.shape)
raw_df.head()


## 2. Chuẩn hóa tên cột và kiểm tra cấu trúc ban đầu

Bước này chuẩn hóa tên cột về dạng `snake_case`, ví dụ `Carrier Name` → `carrier_name`. Việc này giúp code ổn định hơn khi group by, tính toán hoặc export.


In [ ]:
# 2. Chuẩn hóa tên cột

def clean_colname(col):
    col = str(col).strip().lower()
    col = re.sub(r'[^a-z0-9]+', '_', col)
    col = re.sub(r'_+', '_', col).strip('_')
    return col

df = raw_df.copy()
original_columns = list(df.columns)
df.columns = [clean_colname(c) for c in df.columns]

print('Tên cột gốc:')
print(original_columns)
print('
Tên cột sau chuẩn hóa:')
print(list(df.columns))

# Các cột kỳ vọng trong dataset BTS Airline Delay Cause
expected_cols = [
    'year', 'month', 'carrier', 'carrier_name', 'airport', 'airport_name',
    'arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct',
    'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted',
    'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay',
    'security_delay', 'late_aircraft_delay'
]

missing_expected = [c for c in expected_cols if c not in df.columns]
extra_cols = [c for c in df.columns if c not in expected_cols]

print('
Cột kỳ vọng bị thiếu:', missing_expected)
print('Cột ngoài danh sách kỳ vọng:', extra_cols)

if missing_expected:
    print('
CẢNH BÁO: Dataset thiếu một số cột kỳ vọng. Notebook vẫn chạy nếu các cột bắt buộc còn đủ, nhưng cần kiểm tra lại file gốc.')


## 3. Kiểm tra missing values, duplicate và kiểu dữ liệu

Cell này chưa làm sạch ngay, chỉ tạo báo cáo nhanh để biết dataset đang có vấn đề gì.


In [ ]:
# 3. Kiểm tra chất lượng dữ liệu ban đầu
initial_shape = df.shape

missing_before = df.isna().sum().sort_values(ascending=False)
missing_before = missing_before[missing_before > 0]

exact_duplicate_count = int(df.duplicated().sum())
logic_key = ['year', 'month', 'carrier', 'airport']
logic_duplicate_count = None
if all(c in df.columns for c in logic_key):
    logic_duplicate_count = int(df.duplicated(subset=logic_key).sum())

print('Kích thước ban đầu:', initial_shape)
print('
Missing values trước cleaning:')
display(missing_before.to_frame('missing_count'))
print('
Số dòng duplicate hoàn toàn:', exact_duplicate_count)
print('Số dòng duplicate theo khóa year-month-carrier-airport:', logic_duplicate_count)
print('
Kiểu dữ liệu ban đầu:')
display(df.dtypes.to_frame('dtype'))


## 4. Làm sạch dữ liệu

Các bước chính:
1. Xóa dòng trống hoàn toàn.
2. Chuẩn hóa khoảng trắng trong cột text.
3. Ép kiểu các cột số về numeric.
4. Xử lý missing values ở các cột số đếm/phút delay.
5. Xóa duplicate hoàn toàn nếu có.
6. Nếu có duplicate theo khóa logic, gộp lại theo `year-month-carrier-airport`.


In [ ]:
# 4. Làm sạch dữ liệu

# 4.1 Xóa dòng trống hoàn toàn
df = df.dropna(how='all').copy()
rows_after_drop_empty = len(df)

# 4.2 Chuẩn hóa text
text_cols = [c for c in ['carrier', 'carrier_name', 'airport', 'airport_name'] if c in df.columns]
for col in text_cols:
    df[col] = df[col].astype('string').str.strip()
    df[col] = df[col].replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})

# 4.3 Ép kiểu cột số
numeric_cols = [c for c in expected_cols if c not in text_cols and c in df.columns]
for col in numeric_cols:
    # Xử lý trường hợp số có dấu phẩy hoặc ký tự %
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.replace('%', '', regex=False)
        .str.strip()
        .replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 4.4 Xử lý missing ở cột bắt buộc year/month/arr_flights
required_cols = [c for c in ['year', 'month', 'carrier', 'airport', 'arr_flights'] if c in df.columns]
missing_required_mask = df[required_cols].isna().any(axis=1) if required_cols else pd.Series(False, index=df.index)
rows_dropped_missing_required = int(missing_required_mask.sum())
if rows_dropped_missing_required > 0:
    df = df.loc[~missing_required_mask].copy()

# 4.5 Với các cột số đếm và phút delay, missing thường được hiểu là không phát sinh → điền 0
count_delay_cols = [
    'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct',
    'arr_cancelled', 'arr_diverted', 'arr_delay', 'carrier_delay', 'weather_delay',
    'nas_delay', 'security_delay', 'late_aircraft_delay'
]
count_delay_cols = [c for c in count_delay_cols if c in df.columns]
missing_count_delay_before_fill = df[count_delay_cols].isna().sum().to_dict() if count_delay_cols else {}
for col in count_delay_cols:
    df[col] = df[col].fillna(0)

# 4.6 Xử lý giá trị âm nếu có: các biến số chuyến/phút trễ không nên âm → đưa về 0 và lưu log
negative_counts = {}
for col in [c for c in numeric_cols if c in df.columns]:
    neg_mask = df[col] < 0
    negative_counts[col] = int(neg_mask.sum())
    if neg_mask.any():
        df.loc[neg_mask, col] = 0

# 4.7 Xóa duplicate hoàn toàn
exact_duplicate_after_type = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

# 4.8 Nếu trùng khóa logic, gộp lại để mỗi tổ hợp hãng-sân bay-tháng chỉ có 1 dòng
logic_duplicate_after_drop = 0
if all(c in df.columns for c in logic_key):
    logic_duplicate_after_drop = int(df.duplicated(subset=logic_key).sum())
    if logic_duplicate_after_drop > 0:
        agg_dict = {}
        for col in df.columns:
            if col in logic_key:
                continue
            elif col in numeric_cols:
                agg_dict[col] = 'sum'
            else:
                agg_dict[col] = 'first'
        df = df.groupby(logic_key, as_index=False).agg(agg_dict)

print('Số dòng sau cleaning cơ bản:', df.shape[0])
print('Số cột sau cleaning cơ bản:', df.shape[1])
print('Số dòng bị xóa do thiếu cột bắt buộc:', rows_dropped_missing_required)
print('Số duplicate hoàn toàn đã xóa:', exact_duplicate_after_type)
print('Số duplicate logic đã gộp:', logic_duplicate_after_drop)


## 5. Tạo biến phân tích cho bản clean wide

Bước này vẫn giữ dạng wide, nhưng bổ sung các biến hữu ích cho phân tích:
- `period`
- `airport_city`, `airport_state`, `airport_full_name`
- các tỷ lệ: `delay_rate`, `cancel_rate`, `diverted_rate`, `problem_rate`
- các chỉ số phút trễ trung bình
- các tỷ trọng nguyên nhân delay
- các cột kiểm tra độ khớp giữa tổng delay và tổng theo nguyên nhân


In [ ]:
# 5. Tạo biến phân tích

# 5.1 Ép year/month về int nếu có thể
for col in ['year', 'month']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# 5.2 Tạo period YYYY-MM
if all(c in df.columns for c in ['year', 'month']):
    df['period'] = df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2)

# 5.3 Tách airport_name thành city, state, airport_full_name
# Format phổ biến của BTS: "City, ST: Airport Name"
def parse_airport_name(value):
    if pd.isna(value):
        return pd.Series([pd.NA, pd.NA, pd.NA])
    text = str(value).strip()
    if ':' in text:
        left, airport_full = text.split(':', 1)
        airport_full = airport_full.strip()
    else:
        left, airport_full = text, text
    if ',' in left:
        city, state = left.rsplit(',', 1)
        city = city.strip()
        state = state.strip()
    else:
        city, state = left.strip(), pd.NA
    return pd.Series([city, state, airport_full])

if 'airport_name' in df.columns:
    df[['airport_city', 'airport_state', 'airport_full_name']] = df['airport_name'].apply(parse_airport_name)

# 5.4 Hàm chia an toàn
def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors='coerce')
    denominator = pd.to_numeric(denominator, errors='coerce')
    result = numerator / denominator.replace({0: np.nan})
    return result.replace([np.inf, -np.inf], np.nan).fillna(0)

# 5.5 Tạo tỷ lệ chính
if all(c in df.columns for c in ['arr_del15', 'arr_flights']):
    df['delay_rate'] = safe_divide(df['arr_del15'], df['arr_flights'])
if all(c in df.columns for c in ['arr_cancelled', 'arr_flights']):
    df['cancel_rate'] = safe_divide(df['arr_cancelled'], df['arr_flights'])
if all(c in df.columns for c in ['arr_diverted', 'arr_flights']):
    df['diverted_rate'] = safe_divide(df['arr_diverted'], df['arr_flights'])
if all(c in df.columns for c in ['arr_del15', 'arr_cancelled', 'arr_diverted', 'arr_flights']):
    df['problem_rate'] = safe_divide(df['arr_del15'] + df['arr_cancelled'] + df['arr_diverted'], df['arr_flights'])

# 5.6 Tạo chỉ số thời gian delay trung bình
if all(c in df.columns for c in ['arr_delay', 'arr_del15']):
    df['avg_delay_min_per_delayed_flight'] = safe_divide(df['arr_delay'], df['arr_del15'])
if all(c in df.columns for c in ['arr_delay', 'arr_flights']):
    df['avg_delay_min_per_arrival'] = safe_divide(df['arr_delay'], df['arr_flights'])

# 5.7 Tổng theo nguyên nhân và chênh lệch kiểm tra
cause_count_cols = [c for c in ['carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct'] if c in df.columns]
cause_delay_cols = [c for c in ['carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay'] if c in df.columns]

if cause_count_cols:
    df['cause_count_sum'] = df[cause_count_cols].sum(axis=1)
    if 'arr_del15' in df.columns:
        df['cause_count_diff'] = df['arr_del15'] - df['cause_count_sum']

if cause_delay_cols:
    df['cause_delay_min_sum'] = df[cause_delay_cols].sum(axis=1)
    if 'arr_delay' in df.columns:
        df['cause_delay_min_diff'] = df['arr_delay'] - df['cause_delay_min_sum']

# 5.8 Tỷ trọng nguyên nhân theo số lượt và theo phút delay
cause_pairs = {
    'carrier': ('carrier_ct', 'carrier_delay'),
    'weather': ('weather_ct', 'weather_delay'),
    'nas': ('nas_ct', 'nas_delay'),
    'security': ('security_ct', 'security_delay'),
    'late_aircraft': ('late_aircraft_ct', 'late_aircraft_delay')
}

for cause, (ct_col, delay_col) in cause_pairs.items():
    if ct_col in df.columns and 'cause_count_sum' in df.columns:
        df[f'{cause}_ct_share'] = safe_divide(df[ct_col], df['cause_count_sum'])
    if delay_col in df.columns and 'cause_delay_min_sum' in df.columns:
        df[f'{cause}_delay_share'] = safe_divide(df[delay_col], df['cause_delay_min_sum'])

print('Kích thước clean wide:', df.shape)
df.head()


## 6. Kiểm tra nhanh sau cleaning

Cell này giúp xác nhận file clean wide đã ổn để dùng vẽ chart.


In [ ]:
# 6. Kiểm tra sau cleaning
missing_after = df.isna().sum().sort_values(ascending=False)
missing_after = missing_after[missing_after > 0]

print('Kích thước dữ liệu clean wide:', df.shape)
print('
Missing values sau cleaning:')
display(missing_after.to_frame('missing_count'))

if all(c in df.columns for c in ['arr_flights', 'arr_del15', 'arr_cancelled', 'arr_diverted', 'arr_delay']):
    print('
Tổng quan hệ thống:')
    total_flights = df['arr_flights'].sum()
    total_delayed = df['arr_del15'].sum()
    total_cancelled = df['arr_cancelled'].sum()
    total_diverted = df['arr_diverted'].sum()
    total_delay_min = df['arr_delay'].sum()
    print('Tổng chuyến bay đến:', int(total_flights))
    print('Tổng chuyến delay >= 15 phút:', int(total_delayed))
    print('Tổng chuyến hủy:', int(total_cancelled))
    print('Tổng chuyến chuyển hướng:', int(total_diverted))
    print('Tổng phút trễ:', int(total_delay_min))
    print('Delay rate toàn hệ thống:', round(total_delayed / total_flights * 100, 2), '%')

# Kiểm tra logic cơ bản
logic_checks = {}
if all(c in df.columns for c in ['arr_del15', 'arr_flights']):
    logic_checks['arr_del15_gt_arr_flights'] = int((df['arr_del15'] > df['arr_flights']).sum())
if all(c in df.columns for c in ['arr_cancelled', 'arr_flights']):
    logic_checks['arr_cancelled_gt_arr_flights'] = int((df['arr_cancelled'] > df['arr_flights']).sum())
if all(c in df.columns for c in ['arr_diverted', 'arr_flights']):
    logic_checks['arr_diverted_gt_arr_flights'] = int((df['arr_diverted'] > df['arr_flights']).sum())

print('
Logic checks:')
print(logic_checks)


## 7. Xuất file clean wide

File chính cần nộp hoặc dùng cho các chart là `Airline_Delay_Cause_clean_wide.csv`.


In [ ]:
# 7. Export file clean wide và cleaning summary
OUTPUT_DIR = Path('clean_output')
OUTPUT_DIR.mkdir(exist_ok=True)

clean_csv_path = OUTPUT_DIR / 'Airline_Delay_Cause_clean_wide.csv'
summary_json_path = OUTPUT_DIR / 'cleaning_summary.json'
zip_path = 'airline_delay_clean_wide_output.zip'

df.to_csv(clean_csv_path, index=False, encoding='utf-8-sig')

summary = {
    'raw_file': RAW_FILE,
    'initial_shape': list(initial_shape),
    'final_shape': list(df.shape),
    'original_columns': original_columns,
    'clean_columns': list(df.columns),
    'missing_values_before': missing_before.to_dict(),
    'missing_count_delay_before_fill': missing_count_delay_before_fill,
    'missing_values_after': missing_after.to_dict(),
    'exact_duplicate_count_before': exact_duplicate_count,
    'logic_duplicate_count_before': logic_duplicate_count,
    'rows_after_drop_empty': rows_after_drop_empty,
    'rows_dropped_missing_required': rows_dropped_missing_required,
    'exact_duplicates_removed_after_type_cast': exact_duplicate_after_type,
    'logic_duplicates_aggregated': logic_duplicate_after_drop,
    'negative_values_replaced_with_zero': {k: v for k, v in negative_counts.items() if v > 0},
    'logic_checks_after_cleaning': logic_checks,
    'created_file': str(clean_csv_path)
}

with open(summary_json_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(clean_csv_path, arcname=clean_csv_path.name)
    zf.write(summary_json_path, arcname=summary_json_path.name)

print('Đã xuất file clean wide:', clean_csv_path)
print('Đã xuất cleaning summary:', summary_json_path)
print('Đã tạo zip:', zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print('Nếu không chạy trên Colab, hãy tải file trong thư mục:', OUTPUT_DIR)


## 8. Ghi chú dùng trong report

Có thể mô tả ngắn quá trình cleaning như sau:

- Chuẩn hóa tên cột và dữ liệu dạng text.
- Ép các cột số chuyến bay, số chuyến trễ và số phút trễ về kiểu numeric.
- Kiểm tra missing values, duplicate và các lỗi logic cơ bản.
- Điền 0 cho các cột số đếm/phút delay bị thiếu khi không phát sinh delay.
- Tạo thêm các biến phân tích như delay rate, cancellation rate, diversion rate, problem rate và average delay minutes.
- Xuất bản cleaned dataset dạng wide để dùng cho trực quan hóa.
